A company is plannig to launch a feature and they want to decide if they should launch the feature or  not!!

Product manager , Market analyst , Technical analyst , security analyst , financial analyst  

In [ ]:
# Install AG2 (imported as `autogen`) with the Gemini extras, pinned to the classic 0.x API.
# The [gemini] extra pulls in Google's google-genai / google-auth client libraries that AG2
# needs to talk to the Gemini API.
!pip install -q "ag2[gemini]==0.14.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 53.4 MB/s eta 0:00:00


In [ ]:
import sys

MIN_PYTHON = (3, 10)
if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        f"This notebook requires Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]}+ but the current "
        f"runtime is Python {sys.version_info.major}.{sys.version_info.minor}. "
        "In Colab: Runtime > Change runtime type, or use a newer Colab runtime image."
    )

print(f"Python version OK: {sys.version.split()[0]}")

Python version OK: 3.13.15


In [ ]:
# --- Import verification ----------------------------------------------------
# We import every class this notebook will use, right now, so that any incompatible install
# fails immediately in Section 1 -- not halfway through the case study in Section 8.
import autogen  # `autogen` is the import name for the `ag2` package we just installed

from autogen import (
    AssistantAgent,     # an LLM-backed agent that generates replies
    UserProxyAgent,      # an agent that represents "the code/human side" (can execute code/tools)
    ConversableAgent,    # the base class both of the above inherit from
    GroupChat,           # holds a shared conversation between multiple agents
    GroupChatManager,     # orchestrates whose turn it is to speak in a GroupChat
    LLMConfig,           # wraps model + API settings that agents use to call the LLM
)
from autogen.coding import LocalCommandLineCodeExecutor  # runs LLM-generated Python code locally
from autogen.code_utils import content_str                # safely turns a message's content into a string

print("autogen (ag2) version:", autogen.__version__)
print("All required classes imported successfully.")

autogen (ag2) version: 0.14.0
All required classes imported successfully.


In [ ]:
# --- Gemini API key ---------------------------------------------------------
# getpass hides the key as you type it, so it never appears in the notebook output or history.
import getpass

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")

# We ONLY use Google Gemini in this lab. gemini-3.5-flash is a fast, low-cost model that is a
# good fit for a classroom demo with many short agent turns.
GEMINI_MODEL = "gemini-3.5-flash"

# LLMConfig is the object every agent below will be given. "api_type": "google" tells AG2 to
# route requests through its Gemini client instead of OpenAI/Anthropic/etc.
llm_config = LLMConfig({
    "model": GEMINI_MODEL,
    "api_type": "google",
    "api_key": GEMINI_API_KEY,
})

print("LLM config ready for model:", GEMINI_MODEL)

Enter your Gemini API key: ··········
LLM config ready for model: gemini-3.5-flash


Basic agent : one ai agent answers all the questions

In [ ]:
# The "brain": answers questions using Gemini, guided by its system_message.
basic_assistant = AssistantAgent(
    name="Basic_Agent",
    system_message="You are a helpful assistant. Explain AI concepts simply, in 3-4 sentences.",
    llm_config=llm_config,
)

# The "asker": sends one message and does not auto-reply (no LLM needed for this agent).
asker = UserProxyAgent(
    name="Asker",
    human_input_mode="NEVER",        # never pause for real human input
    max_consecutive_auto_reply=0,     # after receiving a reply, do not send anything back
    code_execution_config=False,      # this agent does not run any code
)

# Kick off a one-turn conversation: Asker -> Basic_Agent -> (one reply) -> stop.
basic_chat_result = asker.initiate_chat(
    basic_assistant,
    message="What is an AI agent?",
    max_turns=1,
)

Asker (to Basic_Agent):

What is an AI agent?

--------------------------------------------------------------------------------
Basic_Agent (to Asker):

An AI agent is a smart software program designed to perceive its environment, make decisions, and take action to achieve a specific goal. Unlike basic AI that only answers questions, an agent can work autonomously by planning its own steps, using digital tools, and adapting to new information. For example, instead of just suggesting holiday destinations, an AI travel agent could independently research, book the best flights, and organize your entire itinerary.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (65de2d04-eb65-4c21-9a6f-2b90aae34370): Maximum turns (1) reached


/usr/local/lib/python3.13/dist-packages/autogen/oai/gemini.py:1131: UserWarning: Cost calculation is not implemented for model gemini-3.5-flash. Cost will be calculated zero.
  warnings.warn(


# Two agent conversation


In [ ]:
product_expert = AssistantAgent(
    name="Product_Expert",
    system_message=(
        "You are a product expert at FinTrack. Propose a short (2-3 sentence) idea for how "
        "the AI receipt scanner feature should work for customers."
    ),
    llm_config=llm_config,
)

reviewer = AssistantAgent(
    name="Reviewer",
    system_message=(
        "You are a critical product reviewer. When given a product idea, point out ONE clear "
        "weakness or risk in 2-3 sentences. Be specific, not generic."
    ),
    llm_config=llm_config,
)

# Product_Expert INITIATES the chat with Reviewer, so Product_Expert speaks first with our
# message, and Reviewer replies once (max_turns=1 = one full back-and-forth exchange).
two_agent_result = product_expert.initiate_chat(
    reviewer,
    message="Propose a short idea for FinTrack's AI receipt scanner.",
    max_turns=1,
)

Product_Expert (to Reviewer):

Propose a short idea for FinTrack's AI receipt scanner.

--------------------------------------------------------------------------------
Reviewer (to Product_Expert):

The primary risk lies in the AI's inevitable struggle to accurately parse and categorize abbreviated line items from multi-category retailers (such as "TDR JOE SUB" or "TIDE PODS 38CT" at Target) into a user's specific budget categories. If users constantly have to manually correct misclassified items or split transactions to fix the AI's mistakes, the feature defeats its own time-saving value proposition and will quickly be abandoned.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (69a2b4f6-9eec-4ce5-98d0-a4257c6dffe5): Maximum turns (1) reached


In [ ]:
#Sequential chain
research_agent = AssistantAgent(
    name="Research_Agent",
    system_message="You research markets. Answer with exactly 3 short bullet points, nothing else.",
    llm_config=llm_config,
)

summary_agent = AssistantAgent(
    name="Summary_Agent",
    system_message="You summarize research into ONE concise sentence.",
    llm_config=llm_config,
)

# A plain, non-LLM "runner" agent we reuse just to send single messages and collect replies.
runner = UserProxyAgent(
    name="Runner",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

# Step 1: get research points out of Research_Agent.
research_result = runner.initiate_chat(
    research_agent,
    message="List 3 short bullet points about the market potential for an AI receipt scanner.",
    max_turns=1,
)
research_points = research_result.chat_history[-1]["content"]  # the agent's reply text

print("Research_Agent's output:\n")
print(research_points)


Runner (to Research_Agent):

List 3 short bullet points about the market potential for an AI receipt scanner.

--------------------------------------------------------------------------------
Research_Agent (to Runner):

* **High Demand for Automation:** Huge growth potential driven by SMEs and corporations seeking to eliminate manual data entry and reduce administrative costs.
* **Fintech Integration Opportunities:** Vast market for API integrations with existing accounting, ERP, and expense management software (e.g., QuickBooks, Xero).
* **Regulatory and Remote Drivers:** Accelerated adoption fueled by global tax digitization mandates and the need to track decentralized hybrid workforce expenses.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (1a878901-9756-4fba-81e7-d9682ff67f73): Maximum turns (1) reached
Research_Agent's output:

* **High Demand for Automation:** Huge growth potential driven by SMEs and corporations seek

In [ ]:
# Step 2: feed Research_Agent's OUTPUT directly into Summary_Agent's INPUT message.
summary_result = runner.initiate_chat(
    summary_agent,
    message=f"Summarize the following market research into ONE sentence:\n\n{research_points}",
    max_turns=1,
)
summary_text = summary_result.chat_history[-1]["content"]

print("Summary_Agent's output:\n")
print(summary_text)

Runner (to Summary_Agent):

Summarize the following market research into ONE sentence:

* **High Demand for Automation:** Huge growth potential driven by SMEs and corporations seeking to eliminate manual data entry and reduce administrative costs.
* **Fintech Integration Opportunities:** Vast market for API integrations with existing accounting, ERP, and expense management software (e.g., QuickBooks, Xero).
* **Regulatory and Remote Drivers:** Accelerated adoption fueled by global tax digitization mandates and the need to track decentralized hybrid workforce expenses.

--------------------------------------------------------------------------------
Summary_Agent (to Runner):

Driven by tax digitization and remote workforce needs, businesses are rapidly adopting automated, API-integrated financial tools to eliminate manual data entry and reduce administrative costs.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (f51f2845-14ab

In [ ]:
#Reflection : An agent's output is critiqued and then  agent improves it


writer = AssistantAgent(
    name="Writer",
    system_message="You write short, confident product launch recommendations (1 sentence).",
    llm_config=llm_config,
)

critic = AssistantAgent(
    name="Critic",
    system_message=(
        "You review launch recommendations. In 1-2 sentences, list what important risks or "
        "considerations the recommendation is MISSING (e.g. privacy, accuracy, cost)."
    ),
    llm_config=llm_config,
)

# --- Draft ---
draft_result = runner.initiate_chat(
    writer,
    message="Write a 1-sentence launch recommendation for FinTrack's AI receipt scanner.",
    max_turns=1,
)
draft_text = draft_result.chat_history[-1]["content"]
print("DRAFT:\n", draft_text, "\n")
# --- Critique ---
critique_result = runner.initiate_chat(
    critic,
    message=f"Critique this recommendation:\n\n{draft_text}",
    max_turns=1,
)
critique_text = critique_result.chat_history[-1]["content"]
print("CRITIQUE:\n", critique_text, "\n")

Runner (to Writer):

Write a 1-sentence launch recommendation for FinTrack's AI receipt scanner.

--------------------------------------------------------------------------------
Writer (to Runner):

Launch FinTrack’s AI receipt scanner immediately to capture the market with its high-accuracy automation and eliminate manual expense tracking for users once and for all.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (0e586c1d-e2df-43e5-a780-27a8c02bf603): Maximum turns (1) reached
DRAFT:
 Launch FinTrack’s AI receipt scanner immediately to capture the market with its high-accuracy automation and eliminate manual expense tracking for users once and for all. 

Runner (to Critic):

Critique this recommendation:

Launch FinTrack’s AI receipt scanner immediately to capture the market with its high-accuracy automation and eliminate manual expense tracking for users once and for all.

--------------------------------------------------

#Function calling

In [ ]:
tool_assistant = AssistantAgent(
    name="Finance_Bot",
    system_message=(
        "You are a finance assistant. When asked about ROI, ALWAYS use the calculate_roi tool "
        "instead of calculating by hand. Report the final ROI percentage clearly."
    ),
    llm_config=llm_config,
)

# This agent EXECUTES the tool once Finance_Bot decides to call it.
tool_executor = UserProxyAgent(
    name="Tool_Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,   # allow a couple of automatic back-and-forth tool round trips
    code_execution_config=False,     # no free-form code execution here, only the registered tool
)

# @tool_executor.register_for_execution()   -> "when this tool is called, Tool_Executor runs it"
# @tool_assistant.register_for_llm(...)     -> "tell Finance_Bot's LLM this tool exists"
@tool_executor.register_for_execution()
@tool_assistant.register_for_llm(description="Calculate ROI as a percentage from revenue and cost.")
def calculate_roi(revenue: float, cost: float) -> str:
    roi_percent = ((revenue - cost) / cost) * 100
    return f"{roi_percent:.1f}%"

print("Tool 'calculate_roi' registered on Finance_Bot.")

Tool 'calculate_roi' registered on Finance_Bot.


In [ ]:
#Tool Executor, start a conversation with Finance Bot and give it this ROI question.
tool_result = tool_executor.initiate_chat(
    tool_assistant,
    message="If the receipt scanner is projected to bring in $150,000 revenue at a cost of $50,000, what is the ROI?",
    max_turns=3,
)


Tool_Executor (to Finance_Bot):

If the receipt scanner is projected to bring in $150,000 revenue at a cost of $50,000, what is the ROI?

--------------------------------------------------------------------------------
Finance_Bot (to Tool_Executor):


***** Suggested tool call (7368): calculate_roi *****
Arguments: 
{"revenue": 150000, "cost": 50000}
*****************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION calculate_roi...
Call ID: 7368
Input arguments: {'revenue': 150000, 'cost': 50000}

>>>>>>>> EXECUTED FUNCTION calculate_roi...
Call ID: 7368
Input arguments: {'revenue': 150000, 'cost': 50000}
Output:
200.0%
Tool_Executor (to Finance_Bot):

***** Response from calling tool (7368) *****
200.0%
*********************************************

--------------------------------------------------------------------------------
Finance_Bot (to Tool_Executor):

The ROI for the receipt s

#Group chat

In [ ]:
product_manager = AssistantAgent(
    name="Product_Manager",
    system_message=(
        "You are the Product Manager at FinTrack. You OWN the launch decision. "
        "You are pragmatic, concise, and decision-oriented. You keep the discussion focused, "
        "ask the other analysts for evidence, point out disagreements between them, and near "
        "the end you summarize the discussion and give a final recommendation. "
        "Do NOT simply agree with everyone -- push back when arguments are weak or incomplete. "
        "Keep every message to 2-4 sentences."
    ),
    llm_config=llm_config,
)

market_analyst = AssistantAgent(
    name="Market_Analyst",
    system_message=(
        "You are the Market Analyst. You argue that the AI receipt scanner could reduce manual "
        "receipt entry and improve customer experience, but you are honest that adoption depends "
        "heavily on extraction accuracy and customer trust. Respond directly to points made by "
        "other agents. Keep every message to 2-4 sentences."
    ),
    llm_config=llm_config,
)

technical_analyst = AssistantAgent(
    name="Technical_Analyst",
    system_message=(
        "You are the Technical Analyst. You explain that the feature is technically feasible "
        "using OCR and LLM-based extraction, but receipt formats vary significantly, so extracted "
        "data MUST be validated before being trusted. You flag integration risk with existing "
        "systems. Keep every message to 2-4 sentences."
    ),
    llm_config=llm_config,
)

security_analyst = AssistantAgent(
    name="Security_Analyst",
    system_message=(
        "You are the Security Analyst. Your job is to CHALLENGE the proposal from a privacy and "
        "security angle. Raise concerns about personal and financial information in receipts, "
        "image storage, access control, and data retention. Actively disagree with overly "
        "optimistic claims from other agents when they ignore privacy risk. "
        "Keep every message to 2-4 sentences."
    ),
    llm_config=llm_config,
)

finance_analyst = AssistantAgent(
    name="Finance_Analyst",
    system_message=(
        "You are the Finance Analyst. You evaluate commercial viability: implementation cost, "
        "ongoing AI/API/infrastructure cost, and expected ROI. You challenge the team to justify "
        "business value against operating cost, and suggest a pilot rather than a full launch "
        "when the numbers are uncertain. Keep every message to 2-4 sentences."
    ),
    llm_config=llm_config,
)

print("All 5 case-study agents created.")

All 5 case-study agents created.


In [ ]:
# The GroupChat is the shared "room" all 5 agents are in.
# - agents: everyone allowed to participate
# - max_round: hard cap on total turns, so the discussion can't run away (keeps API calls low)
# - speaker_selection_method="round_robin": agents speak in a fixed, repeating order --
#   simple and predictable for a live class
groupchat = GroupChat(
    agents=[product_manager, market_analyst, technical_analyst, security_analyst, finance_analyst],
    messages=[],
    max_round=11,
    speaker_selection_method="round_robin",
)

# The GroupChatManager is the "facilitator": it receives each agent's message and passes the
# floor to the next speaker according to the GroupChat's rules above.
manager = GroupChatManager(
    name="chat_manager",
    groupchat=groupchat,
    llm_config=llm_config,
)

print("GroupChat ready with", len(groupchat.agents), "agents, max_round =", groupchat.max_round)

GroupChat ready with 5 agents, max_round = 11


In [ ]:
CASE_STUDY_PROMPT = (
    "FinTrack is considering launching an AI-powered receipt scanner next quarter. The feature "
    "lets customers upload or photograph a receipt, and AI extracts the merchant, date, items, "
    "total, tax, and spending category. Product_Manager, please open the discussion: should "
    "FinTrack launch it? Discuss customer value, technical feasibility, security/privacy, and "
    "cost -- then summarize with a final recommendation."
)

# Product_Manager sends the opening prompt into the GroupChat via the manager.
# From here, GroupChatManager automatically rotates speakers (round-robin) for up to
# groupchat.max_round turns, with each agent reading the FULL shared history so far.
case_study_result = product_manager.initiate_chat(
    manager,
    message=CASE_STUDY_PROMPT,
)

Product_Manager (to chat_manager):

FinTrack is considering launching an AI-powered receipt scanner next quarter. The feature lets customers upload or photograph a receipt, and AI extracts the merchant, date, items, total, tax, and spending category. Product_Manager, please open the discussion: should FinTrack launch it? Discuss customer value, technical feasibility, security/privacy, and cost -- then summarize with a final recommendation.

--------------------------------------------------------------------------------

Next speaker: Market_Analyst

Market_Analyst (to chat_manager):

From a market perspective, this feature addresses a major user friction point by eliminating tedious manual receipt entry, which could significantly boost our customer engagement. However, market adoption is entirely contingent on extraction accuracy; if the AI constantly misinterprets totals or merchants, users will quickly abandon the tool out of frustration. Furthermore, building customer trust around 